# Baseline EU YOLO Model
Builds a YOLO dataset from Mapillary EU signs only (no LISA) and trains a baseline detector.

In [3]:
import os
import csv
import json
import yaml
import shutil
import random
import numpy as np
from pathlib import Path
from collections import defaultdict
from PIL import Image
from tqdm.notebook import tqdm

## Configuration

In [5]:
# ── Inputs ────────────────────────────────────────────────────────────────────
MAPILLARY_ROOT  = Path("dataset/mtsd_v2_fully_annotated")
MAPILLARY_CSV   = Path("mapillary_class_review_eu.csv")  # EU review CSV
TEST_HOLDOUT    = 0.10
USE_TILING      = False   # set True to tile high-res images to 640px crops

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT     = Path("dataset/baseline_eu_v1")
PROJECT_DIR = os.path.abspath("experiments/")
RUN_NAME    = "baseline_eu_v1"

# ── Annotation filtering ──────────────────────────────────────────────────────
MIN_ANNOTATION_PX = 8    # drop annotations smaller than this in either dimension

SEED = 42
random.seed(SEED)
print(f"Output: {OUTPUT_ROOT}")

Output: dataset\baseline_eu_v1


## Step 1 — Build class list from EU CSV

In [3]:
# ── Class merges — fold variant g-numbers into a single canonical label ────────
# Add any EU-specific merges here once you've reviewed the confusion matrix.
# Leave empty for the first training run.
MAPILLARY_MERGES = {
    # Example: "warning--priority-road--g1": ["warning--priority-road--g2"],
}


def apply_merges(kept_labels, merges):
    remap = {}
    for canonical, variants in merges.items():
        for variant in variants:
            if variant in kept_labels:
                remap[variant] = canonical
                kept_labels.discard(variant)
        if canonical in kept_labels:
            remap[canonical] = canonical
    for label in kept_labels:
        if label not in remap:
            remap[label] = label
    return kept_labels, remap


def load_mapillary_kept(csv_path):
    kept = set()
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            if row["keep"] == "yes":
                kept.add(row["label"])
    return kept


mapillary_kept = load_mapillary_kept(MAPILLARY_CSV)
mapillary_kept, mapillary_remap = apply_merges(mapillary_kept, MAPILLARY_MERGES)

all_labels  = sorted(mapillary_kept)
label_to_id = {label: i for i, label in enumerate(all_labels)}

print(f"EU classes: {len(all_labels)}")
for label in all_labels:
    print(f"  {label_to_id[label]:3d}  {label}")

EU classes: 51
    0  information--pedestrians-crossing--g1
    1  regulatory--bicycles-only--g1
    2  regulatory--go-straight--g1
    3  regulatory--go-straight-or-turn-left--g1
    4  regulatory--go-straight-or-turn-right--g1
    5  regulatory--keep-left--g1
    6  regulatory--keep-right--g1
    7  regulatory--maximum-speed-limit-10--g1
    8  regulatory--maximum-speed-limit-100--g1
    9  regulatory--maximum-speed-limit-110--g1
   10  regulatory--maximum-speed-limit-120--g1
   11  regulatory--maximum-speed-limit-20--g1
   12  regulatory--maximum-speed-limit-30--g1
   13  regulatory--maximum-speed-limit-40--g1
   14  regulatory--maximum-speed-limit-45--g1
   15  regulatory--maximum-speed-limit-50--g1
   16  regulatory--maximum-speed-limit-60--g1
   17  regulatory--maximum-speed-limit-70--g1
   18  regulatory--maximum-speed-limit-80--g1
   19  regulatory--maximum-speed-limit-90--g1
   20  regulatory--no-entry--g1
   21  regulatory--no-heavy-goods-vehicles--g2
   22  regulatory--no-le

## Step 2 — Set up output directory

In [4]:
if OUTPUT_ROOT.exists():
    print(f"Clearing existing {OUTPUT_ROOT}...")
    shutil.rmtree(OUTPUT_ROOT)

for split in ["train", "val", "test"]:
    (OUTPUT_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_ROOT.resolve()}")

Output directory: C:\Projects\TrafficSignRecognition\model_training\dataset\baseline_eu_v1


## Step 3 — Process Mapillary

In [5]:
def load_mapillary_keys(mapillary_root, splits, test_holdout=0.10, seed=42):
    key_to_split = {}
    for split in splits:
        path = mapillary_root / "splits" / f"{split}.txt"
        if not path.exists():
            continue
        with open(path) as f:
            for line in f:
                key = line.strip()
                if key:
                    # Mapillary test split has no labels — redirect to val
                    key_to_split[key] = "val" if split == "test" else split

    train_keys = [k for k, s in key_to_split.items() if s == "train"]
    random.seed(seed)
    random.shuffle(train_keys)
    n_test = int(len(train_keys) * test_holdout)
    for key in train_keys[:n_test]:
        key_to_split[key] = "test"

    from collections import Counter
    counts = Counter(key_to_split.values())
    print(f"  Mapillary split assignment:")
    print(f"    train : {counts['train']:,}")
    print(f"    val   : {counts['val']:,}  (original val + redirected test)")
    print(f"    test  : {counts['test']:,}  ({test_holdout:.0%} held out from train)")
    return key_to_split


def mapillary_to_yolo(bbox, img_w, img_h):
    xmin, ymin = bbox["xmin"], bbox["ymin"]
    xmax, ymax = bbox["xmax"], bbox["ymax"]
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    w  = (xmax - xmin) / img_w
    h  = (ymax - ymin) / img_h
    return cx, cy, w, h


def is_too_small(bbox, min_px):
    return (bbox["xmax"] - bbox["xmin"]) < min_px or \
           (bbox["ymax"] - bbox["ymin"]) < min_px


def get_tiles(img_w, img_h, tile_size=640, overlap=0.2):
    step  = int(tile_size * (1 - overlap))
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1     = min(y0 + tile_size, img_h)
        y0_adj = max(0, y1 - tile_size)
        x0 = 0
        while x0 < img_w:
            x1     = min(x0 + tile_size, img_w)
            x0_adj = max(0, x1 - tile_size)
            tiles.append((x0_adj, y0_adj, x0_adj + tile_size, y0_adj + tile_size))
            if x1 == img_w: break
            x0 += step
        if y1 == img_h: break
        y0 += step
    return list(dict.fromkeys(tiles))


def remap_annotations_to_tile(yolo_lines, tile_x0, tile_y0, tile_size,
                               img_w, img_h, min_ann_px=10):
    tile_x1 = tile_x0 + tile_size
    tile_y1 = tile_y0 + tile_size
    tile_anns = []
    for line in yolo_lines:
        parts          = line.strip().split()
        cls_id         = parts[0]
        cx, cy, bw, bh = [float(x) for x in parts[1:5]]
        pcx = cx * img_w;  pcy = cy * img_h
        pbw = bw * img_w;  pbh = bh * img_h
        if not (tile_x0 <= pcx < tile_x1 and tile_y0 <= pcy < tile_y1):
            continue
        pxmin = max(pcx - pbw/2, tile_x0)
        pymin = max(pcy - pbh/2, tile_y0)
        pxmax = min(pcx + pbw/2, tile_x1)
        pymax = min(pcy + pbh/2, tile_y1)
        if (pxmax - pxmin) < min_ann_px or (pymax - pymin) < min_ann_px:
            continue
        new_cx = max(0.0, min(1.0, (pcx - tile_x0) / tile_size))
        new_cy = max(0.0, min(1.0, (pcy - tile_y0) / tile_size))
        new_bw = max(0.0, min(1.0, (pxmax - pxmin) / tile_size))
        new_bh = max(0.0, min(1.0, (pymax - pymin) / tile_size))
        tile_anns.append(f"{cls_id} {new_cx:.6f} {new_cy:.6f} {new_bw:.6f} {new_bh:.6f}")
    return tile_anns


def process_mapillary(mapillary_root, kept_labels, label_to_id,
                      output_root, mapillary_remap=None,
                      min_annotation_px=8, use_tiling=False):
    ann_dir = mapillary_root / "annotations"
    img_dir = mapillary_root / "images"

    key_to_split = load_mapillary_keys(
        mapillary_root, ["train", "val", "test"],
        test_holdout=TEST_HOLDOUT, seed=SEED,
    )
    split_remap = {"train": "train", "val": "val", "test": "test"}

    if mapillary_remap is None:
        mapillary_remap = {label: label for label in kept_labels}

    stats             = defaultdict(lambda: defaultdict(int))
    skipped_no_image  = 0
    skipped_no_labels = 0
    skipped_pano      = 0
    skipped_too_small = 0
    skipped_occluded  = 0
    written           = 0

    ann_files = list(ann_dir.glob("*.json"))

    for ann_path in tqdm(ann_files, desc="Mapillary"):
        key      = ann_path.stem
        img_path = img_dir / f"{key}.jpg"
        if not img_path.exists():
            skipped_no_image += 1
            continue

        with open(ann_path) as f:
            ann = json.load(f)

        if ann.get("ispano", False):
            skipped_pano += 1
            continue

        img_w = ann["width"]
        img_h = ann["height"]

        yolo_lines = []
        for obj in ann.get("objects", []):
            raw_label = obj.get("label", "")
            if raw_label not in mapillary_remap:
                continue

            props = obj.get("properties", {})
            if props.get("ambiguous") or props.get("out-of-frame") or props.get("dummy"):
                continue
            if props.get("occluded"):
                skipped_occluded += 1
                continue
            if is_too_small(obj["bbox"], min_annotation_px):
                skipped_too_small += 1
                continue

            label    = mapillary_remap[raw_label]
            class_id = label_to_id[label]
            cx, cy, w, h = mapillary_to_yolo(obj["bbox"], img_w, img_h)

            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            w  = max(0.0, min(1.0, w))
            h  = max(0.0, min(1.0, h))

            yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            split_out = split_remap.get(key_to_split.get(key, "train"), "train")
            stats[split_out][label] += 1

        if not yolo_lines:
            skipped_no_labels += 1
            continue

        split_out = split_remap.get(key_to_split.get(key, "train"), "train")

        if use_tiling:
            img      = Image.open(img_path).convert("RGB")
            tiles    = get_tiles(img_w, img_h, tile_size=640, overlap=0.2)
            wrote_any = False
            for t_idx, (x0, y0, x1, y1) in enumerate(tiles):
                tile_anns = remap_annotations_to_tile(
                    yolo_lines, x0, y0, 640, img_w, img_h, min_ann_px=10
                )
                if not tile_anns:
                    continue
                stem = f"mply_{key}_t{t_idx}"
                img.crop((x0, y0, x1, y1)).save(
                    output_root / split_out / "images" / f"{stem}.jpg", quality=92)
                (output_root / split_out / "labels" / f"{stem}.txt").write_text(
                    "\n".join(tile_anns))
                wrote_any = True
            if wrote_any:
                written += 1
        else:
            shutil.copy2(img_path, output_root / split_out / "images" / f"mply_{key}.jpg")
            (output_root / split_out / "labels" / f"mply_{key}.txt").write_text(
                "\n".join(yolo_lines))
            written += 1

    print(f"  Written            : {written:,} images")
    print(f"  Skipped (no image) : {skipped_no_image:,}")
    print(f"  Skipped (pano)     : {skipped_pano:,}")
    print(f"  Skipped (no kept labels) : {skipped_no_labels:,}")
    print(f"  Annotations dropped (too small) : {skipped_too_small:,}")
    print(f"  Annotations dropped (occluded)  : {skipped_occluded:,}")
    return stats


print("Processing Mapillary...")
mapillary_stats = process_mapillary(
    MAPILLARY_ROOT, mapillary_kept, label_to_id,
    OUTPUT_ROOT,
    mapillary_remap    = mapillary_remap,
    min_annotation_px  = MIN_ANNOTATION_PX,
    use_tiling         = USE_TILING,
)
print("Done.")

Processing Mapillary...
  Mapillary split assignment:
    train : 32,931
    val   : 15,864  (original val + redirected test)
    test  : 3,658  (10% held out from train)


Mapillary:   0%|          | 0/41909 [00:00<?, ?it/s]

  Written            : 13,864 images
  Skipped (no image) : 0
  Skipped (pano)     : 941
  Skipped (no kept labels) : 27,104
  Annotations dropped (too small) : 21
  Annotations dropped (occluded)  : 1,813
Done.


## Step 4 — Write data.yaml

In [6]:
data_yaml = {
    "path":  str(OUTPUT_ROOT.resolve()),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    len(all_labels),
    "names": all_labels,
}

yaml_out = OUTPUT_ROOT / "data.yaml"
with open(yaml_out, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"data.yaml written to {yaml_out}")

data.yaml written to dataset\baseline_eu_v1\data.yaml


## Step 5 — Dataset Summary

In [7]:
def count_files(path):
    return len(list(path.glob("*"))) if path.exists() else 0

combined_counts = defaultdict(int)
for split_counts in mapillary_stats.values():
    for label, count in split_counts.items():
        combined_counts[label] += count

arr = np.array([combined_counts.get(l, 0) for l in all_labels])

print("=" * 70)
print("EU DATASET SUMMARY")
print("=" * 70)
print(f"  Total classes : {len(all_labels)}")
print()
for split in ["train", "val", "test"]:
    n = count_files(OUTPUT_ROOT / split / "images")
    print(f"  {split:<6} images : {n:,}")
print()
print(f"  Total annotations : {arr.sum():,}")
print(f"  Mean per class    : {arr.mean():,.1f}")
print(f"  Median per class  : {np.median(arr):,.1f}")
print(f"  Classes < 100     : {(arr < 100).sum()}")
print(f"  Classes < 500     : {(arr < 500).sum()}")
print()
print(f"  {'Class':<55} {'Total':>7}")
print("  " + "─" * 64)
for label in all_labels:
    total = combined_counts.get(label, 0)
    flag  = "  ⚠️ <100" if total < 100 else ""
    print(f"  {label:<55} {total:>7,}{flag}")
print("=" * 70)

EU DATASET SUMMARY
  Total classes : 51

  train  images : 10,919
  val    images : 1,738
  test   images : 1,207

  Total annotations : 24,283
  Mean per class    : 476.1
  Median per class  : 286.0
  Classes < 100     : 3
  Classes < 500     : 39

  Class                                                     Total
  ────────────────────────────────────────────────────────────────
  information--pedestrians-crossing--g1                     1,944
  regulatory--bicycles-only--g1                               267
  regulatory--go-straight--g1                                 423
  regulatory--go-straight-or-turn-left--g1                    145
  regulatory--go-straight-or-turn-right--g1                   126
  regulatory--keep-left--g1                                   402
  regulatory--keep-right--g1                                1,123
  regulatory--maximum-speed-limit-10--g1                       84  ⚠️ <100
  regulatory--maximum-speed-limit-100--g1                     167
  regulatory--

## Training

In [ ]:
from ultralytics import YOLO
import os

model = YOLO("yolov8s.pt")

results = model.train(
    # ── Core ──────────────────────────────────────────────────────
    data     = str(OUTPUT_ROOT / "data.yaml"),
    epochs   = 150,
    patience = 30,
    imgsz    = 640,
    batch    = 16,
    device   = 0,

    # ── Output ────────────────────────────────────────────────────
    project  = PROJECT_DIR,
    name     = RUN_NAME,
    exist_ok = False,

    # ── Optimizer ─────────────────────────────────────────────────
    optimizer     = "SGD",
    lr0           = 0.01,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3.0,
    warmup_momentum = 0.8,
    cos_lr        = True,

    # ── Augmentation ──────────────────────────────────────────────
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 5.0,
    translate   = 0.1,
    scale       = 0.7,
    shear       = 2.0,
    perspective = 0.0003,
    flipud      = 0.0,
    fliplr      = 0.0,   # disabled — directional signs must not be flipped
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.3,
    erasing     = 0.4,

    # ── Evaluation ────────────────────────────────────────────────
    val         = True,
    save        = True,
    save_period = 10,
    plots       = True,
)

New https://pypi.org/project/ultralytics/8.4.30 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=dataset\baseline_eu_v1\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.

## Evaluate on Test Set

In [6]:
from ultralytics import YOLO

model = YOLO(f"experiments/{RUN_NAME}/weights/best.pt")

metrics = model.val(
    data    = str(OUTPUT_ROOT / "data.yaml"),
    split   = "test",
    imgsz   = 640,
    device  = 0,
    plots   = True,
    save_json = False,
)

print(f"mAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
Model summary (fused): 73 layers, 11,145,321 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 200.6129.2 MB/s, size: 753.3 KB)
val: Scanning C:\Projects\TrafficSignRecognition\model_training\dataset\baseline_eu_v1\test\labels... 1207 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1207/1207 965.4it/s 1.3s0.0s
val: New cache created: C:\Projects\TrafficSignRecognition\model_training\dataset\baseline_eu_v1\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 8.4it/s 9.1s0.1s
                   all       1207       2104      0.628      0.428      0.488      0.368
information--pedestrians-crossing--g1        126        195      0.687      0.779      0.782      0.548
regulatory--bicycles-only--g1         20         24      0.692      0.333      0.443      0.336
regulator

## Confusion Analysis

In [7]:
from ultralytics import YOLO
from collections import defaultdict

model = YOLO(f"experiments/{RUN_NAME}/weights/best.pt")
class_names = list(model.names.values())

results_val = model.val(
    data    = str(OUTPUT_ROOT / "data.yaml"),
    split   = "val",
    imgsz   = 640,
    device  = 0,
    plots   = False,
)

# Per-class mAP
print(f"\n{'Class':<55} {'mAP50':>7}")
print("─" * 64)
for i, (cls_name, map_val) in enumerate(
        zip(class_names, results_val.box.maps)):
    flag = "  ⚠️" if map_val < 0.5 else ""
    print(f"  {cls_name:<53} {map_val:.3f}{flag}")

# Confusion matrix — top confused pairs
conf_matrix = results_val.confusion_matrix.matrix
pairs = []
for true_idx in range(len(class_names)):
    for pred_idx in range(len(class_names)):
        if true_idx == pred_idx:
            continue
        count = int(conf_matrix[pred_idx][true_idx])
        if count >= 3:
            true_total = int(conf_matrix[:, true_idx].sum())
            rate = count / true_total if true_total else 0
            pairs.append((class_names[true_idx], class_names[pred_idx],
                          count, rate))

pairs.sort(key=lambda x: -x[2])

print(f"\n{'='*80}")
print(f"  TOP CONFUSED CLASS PAIRS  (min 3 instances)")
print(f"{'='*80}")
print(f"  {'True Class':<45} {'Predicted As':<45} {'Count':>6}  {'Rate':>6}")
print(f"  {'─'*45} {'─'*45} {'─'*6}  {'─'*6}")
for true_cls, pred_cls, count, rate in pairs[:30]:
    print(f"  {true_cls:<45} {pred_cls:<45} {count:>6}  {rate:>5.1%}")

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
Model summary (fused): 73 layers, 11,145,321 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 3225.3443.0 MB/s, size: 1036.9 KB)
val: Scanning C:\Projects\TrafficSignRecognition\model_training\dataset\baseline_eu_v1\val\labels.cache... 1738 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1738/1738  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 109/109 9.6it/s 11.4s0.1ss
                   all       1738       3079      0.642      0.424      0.485      0.362
information--pedestrians-crossing--g1        158        242      0.703      0.773      0.793      0.538
regulatory--bicycles-only--g1         30         33      0.502      0.394      0.404       0.32
regulatory--go-straight--g1         47         54      0.729      0.349      0.443      0.317
regulatory--go-straight-or-turn-

## Export to ONNX

In [8]:
from ultralytics import YOLO
from pathlib import Path
import onnx

MODEL_PATH   = f"experiments/{RUN_NAME}/weights/best.pt"
EXPORT_IMGSZ = 640

model = YOLO(MODEL_PATH)
export_path = model.export(
    format   = "onnx",
    imgsz    = EXPORT_IMGSZ,
    half     = False,
    dynamic  = False,
    simplify = True,
)

# Inline weights so there's no separate .onnx.data file
model_proto = onnx.load(str(export_path))
onnx.save_model(model_proto, str(export_path), save_as_external_data=False)

print(f"Exported to : {export_path}")
print(f"File size   : {Path(export_path).stat().st_size / 1e6:.1f} MB")

Ultralytics 8.4.21  Python-3.11.9 torch-2.10.0+cu130 CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
Model summary (fused): 73 layers, 11,145,321 parameters, 0 gradients, 28.5 GFLOPs

PyTorch: starting from 'experiments\baseline_eu_v1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 55, 8400) (21.5 MB)

ONNX: starting export with onnx 1.20.1 opset 22...


c:\Projects\TrafficSignRecognition\model_training\.venv\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(


ONNX: slimming with onnxslim 0.1.89...
ONNX: export success  1.1s, saved as 'experiments\baseline_eu_v1\weights\best.onnx' (42.7 MB)

Export complete (1.4s)
Results saved to C:\Projects\TrafficSignRecognition\model_training\experiments\baseline_eu_v1\weights
Predict:         yolo predict task=detect model=experiments\baseline_eu_v1\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=experiments\baseline_eu_v1\weights\best.onnx imgsz=640 data=dataset\baseline_eu_v1\data.yaml  
Visualize:       https://netron.app
Exported to : experiments\baseline_eu_v1\weights\best.onnx
File size   : 44.8 MB


## Generate Class Files For Android

In [9]:
import json

assets_dir = Path(f"experiments/{RUN_NAME}/android_assets")
assets_dir.mkdir(exist_ok=True)

with open(OUTPUT_ROOT / "data.yaml") as f:
    data = yaml.safe_load(f)

# Plain text — one class per line, index = line number
with open(assets_dir / "eu_classes.txt", "w") as f:
    for name in data["names"]:
        f.write(name + "\n")

# JSON
with open(assets_dir / "eu_classes.json", "w") as f:
    json.dump({i: name for i, name in enumerate(data["names"])}, f, indent=2)

print(f"{len(data['names'])} EU classes exported to {assets_dir}")

51 EU classes exported to experiments\baseline_eu_v1\android_assets
